# Edge Development Tooling · SSH, Linux, Git, and GitHub

This is the first hands-on exercise of the course. It sets up the tools you will use for every later exercise: a secure connection to the edge device, the Linux command line, and version control. (Containers with Docker come in the next exercise, System Architecture & Design.)

The goal is to practice the everyday remote-development workflow used to build and operate edge systems:

```text
Your Computer -> SSH -> Jetson Thor -> Linux + Git -> GitHub
```

You will:

- connect to a Jetson Thor edge device with SSH
- practice Linux commands on the device
- track files with Git and push them to GitHub over SSH

The main idea is that **SSH** is the secure connection layer, **Linux** is the environment where commands run, and **Git/GitHub** track and share project history. The next exercise adds **Docker** to package and run services on the edge device.

## How this notebook works · three places to run things

This notebook runs in JupyterLab **on the Jetson Thor itself**. That creates a twist unique to this first lab: Parts 1-6 set up the SSH connection *from your own computer* to the Thor, so those commands cannot run as cells here — they belong on your laptop.

Three labels are used throughout:

- **[Notebook cell]** · run the cell right here. It executes on the Thor.
- **[Terminal]** · open a Jupyter terminal on the Thor — **File ▸ New ▸ Terminal** (or the blue **+** Launcher, then the **Terminal** tile) — and run the command there. Used for interactive steps.
- **[Your computer · Terminal]** · open a terminal app on your own laptop (Terminal on macOS/Linux; PowerShell, WSL, or Git Bash on Windows) and run the command there. Used in Parts 1-6 and 15 only.

Two useful consequences of running on the Thor:

- the notebook can **verify your laptop-side work from the Thor's side** — when your public key lands in `~/.ssh/authorized_keys` here, the Parts 1-5 checkpoint turns green
- each `!` line in a code cell is its own tiny shell, so persistent directory changes use `%cd` instead of `cd`

Later labs use a shared `labEnv.sh` config file for assigned ports; this lab needs no ports, so there is none here.

In [1]:
# Load the shared lab toolkit (labHelpers.py ships in the course repo next to
# this notebook). It provides pretty output, preflight checks, and checkpoints.
import sys, pathlib
searchDirs = [pathlib.Path.cwd(), *list(pathlib.Path.cwd().parents)[:3],
              pathlib.Path.home() / "EdgeClassHandson"]
helperDir = next((d for d in searchDirs if (d / "labHelpers.py").exists()), None)
assert helperDir is not None, "labHelpers.py not found - keep it next to this notebook"
sys.path.insert(0, str(helperDir))
from labHelpers import *

labHelpers ready - setupLab, preflight, checkpoint, labSummary, showFile, showScriptCard, showEnvCard, showNote, dockerVersions, dockerPs, dockerLogs + check predicates


---
## Part 0 · About the Device

The whole class shares **one Jetson Thor**. Each student or group has their own Linux account on that single device. Because the device is shared:

- use your own account, your own home folder, and your own working folders
- prefix any shared resource names with your account name using `${USER}`

This is the only hands-on exercise where you set up the SSH connection itself. In every later hands-on exercise you will already be connected to the Thor with your own account, so those exercises start directly on the device.

### Preflight · check your environment

**[Notebook cell]** Run this once at the start. It confirms the tools the Thor-side parts need (Git for Parts 9-14, OpenSSH for Part 11) and shows the account this notebook is running as. Fix any failing check before continuing.

In [2]:
import os
preflight([
    check("git installed on the Thor", commandOnPath("git"),
          hint="Git is needed from Part 9 onward. Ask your instructor to install it "
               "(sudo apt install git) - it normally ships with the Thor's OS image.",
          link="https://git-scm.com/book/en/v2/Getting-Started-Installing-Git",
          linkText="Installing Git"),
    check("ssh client on the Thor", commandOnPath("ssh"),
          hint="OpenSSH is needed in Part 11 so the Thor can talk to GitHub. "
               "Ask your instructor if it is missing.",
          link="https://www.openssh.com/manual.html",
          linkText="OpenSSH manual"),
    check("ssh-keygen available", commandOnPath("ssh-keygen"),
          hint="ssh-keygen creates the Thor's GitHub key in Part 11; it ships with OpenSSH.",
          link="https://man.openbsd.org/ssh-keygen",
          linkText="ssh-keygen man page"),
], infoRows=[("your USER", os.environ.get("USER", "?")),
             ("your home folder", os.path.expanduser("~"))])

                   preflight - environment                    
                                                              
  check                       result   detail                 
 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 
  git installed on the Thor   ✓ ok     /opt/homebrew/bin/git  
  ssh client on the Thor      ✓ ok     /usr/bin/ssh           
  ssh-keygen available        ✓ ok     /usr/bin/ssh-keygen    
  your USER                   info     papka                  
  your home folder            info     /Users/papka

True

---
## Part 1 · Set Your Thor Account and Working Folder

> The commands in Parts 1 through 6 run **on your own computer**, because SSH connects *from* your computer *to* the Thor. Starting at "You Are Now Working On the Thor" you switch back to notebook cells.

Your instructor gives you two things: the Thor's **address** and your **account username**. Save them as environment variables now, so that for the rest of this exercise you connect with a single `ssh ${DEVICE_USER}@${DEVICE_ADDRESS}` command instead of retyping them.

**[Your computer · Terminal]** Set the Thor's address and your account username (replace the values with what your instructor gave you), then check that both saved correctly:

```bash
export DEVICE_ADDRESS=<thor-address>
export DEVICE_USER=<your-account>
echo $DEVICE_USER
echo $DEVICE_ADDRESS
```

**[Your computer · Terminal]** Create a local working folder on your own computer:

```bash
mkdir -p ~/edgeToolingLab
cd ~/edgeToolingLab
```

---
## Part 2 · Verify SSH and Reach the Thor

**[Your computer · Terminal]** Check that the SSH client is installed on your computer:

```bash
ssh -V
```

Test whether the device is reachable (stop the ping with **Ctrl-C**):

```bash
ping $DEVICE_ADDRESS
```

If ping is blocked on the network, SSH may still work. Try connecting:

```bash
ssh ${DEVICE_USER}@${DEVICE_ADDRESS}
```

If this is your first time connecting, SSH will show a message about the host authenticity. Type `yes`, then enter the password provided by the instructor.

After logging in, run:

```bash
whoami
hostname
pwd
```

**You should see:** your shell prompt changes (it now shows the Thor's hostname), `whoami` prints your assigned username, and `hostname` prints the Thor's name — confirming the commands are running on the device, not your laptop.

> **If it doesn't work:**
> - `Connection refused` / `timed out` → wrong address, the Thor is off, or you're not on the right network. Re-check `echo $DEVICE_ADDRESS` and ask your instructor.
> - `Permission denied` → wrong username or password. Confirm `echo $DEVICE_USER` and that you're typing the instructor-provided password (terminals show nothing as you type a password — that is normal).
> - `command not found: ssh` → install an SSH client (macOS/Linux have it; on Windows use WSL or Git Bash).

Exit the device:

```bash
exit
```

This shows that SSH opens a secure terminal session on another device.

---
## Part 3 · Understand What SSH Is Doing

SSH stands for Secure Shell. It is commonly used to:

- log in to a remote Linux device
- run commands on a remote device
- copy files securely
- connect Git to remote repositories
- manage servers, edge devices, and cloud machines

The basic SSH command has this structure:

```text
ssh username@device_address
```

With the variables you set in Part 1, that is exactly the command you already ran:

```text
${DEVICE_USER}     -> the Linux account to log in as
${DEVICE_ADDRESS}  -> the Thor's address
ssh                -> the secure connection program
```

SSH checks two identities: the **remote device identity** (stored on your computer in `~/.ssh/known_hosts`) and **your user identity** (checked with a password or an SSH key).

**[Your computer · Terminal]** View your SSH folder:

```bash
ls -la ~/.ssh
```

If the folder does not exist yet, create it:

```bash
mkdir -p ~/.ssh
chmod 700 ~/.ssh
```

The `chmod 700` command means only your user can read, write, and open the folder.

---
## Part 4 · Create and Use an SSH Key

An SSH key is a pair of files:

```text
private key -> stays on your computer
public key  -> can be copied to servers and Git hosts
```

Important:

```text
Never share your private key.
The private key proves your identity.
The public key is safe to copy to systems that need to recognize you.
```

**[Your computer · Terminal]** Create an SSH key for this hands-on exercise on your computer:

```bash
ssh-keygen -t ed25519 -C "${USER}-edgeToolingLab"
```

When asked for the file location, press Enter to use the default (`~/.ssh/id_ed25519`). When asked for a passphrase, use the instructor's guidance.

Check that the key files exist, then view the public key:

```bash
ls -la ~/.ssh/id_ed25519*
cat ~/.ssh/id_ed25519.pub
```

The `.pub` file is the public key, and the file without `.pub` is the private key. Do not print or share the private key.

---
## Part 5 · Copy Your SSH Key to the Thor

In this part you set up passwordless SSH login to the Thor.

**[Your computer · Terminal]** Use `ssh-copy-id` if it is available (enter your Linux account password when prompted):

```bash
ssh-copy-id ${DEVICE_USER}@${DEVICE_ADDRESS}
```

Now try connecting again:

```bash
ssh ${DEVICE_USER}@${DEVICE_ADDRESS}
```

If the key was copied correctly, SSH should let you in without asking for the account password. Exit the device with `exit`.

> **If `ssh-copy-id` is not available**, do the same thing by hand: `cat ~/.ssh/id_ed25519.pub` on your computer, SSH into the Thor, and append that line to `~/.ssh/authorized_keys` (create `~/.ssh` with `chmod 700` and the file with `chmod 600` first). `ssh-copy-id` just automates exactly this — appending your public key to the server's `authorized_keys`.

This shows how SSH public-key authentication works: the server recognizes any key listed in its `authorized_keys`, so you log in without a password.

### Checkpoint · Parts 1-5

This notebook runs **on the Thor**, so it can see whether your laptop's key actually arrived. Parts 1-4 happen entirely on your computer (self-check: variables set, ping or ssh reached the device, key pair created) — Part 5 is what leaves a verifiable trace here.

In [3]:
checkpoint("Parts 1-5 - SSH key reaches the Thor", [
    check("~/.ssh exists on the Thor", dirExists("~/.ssh"),
          hint="No ~/.ssh folder in your Thor account yet - run Part 5's ssh-copy-id "
               "from your own computer (it creates the folder for you).",
          link="https://man.openbsd.org/ssh-copy-id",
          linkText="ssh-copy-id man page"),
    check("your public key is in authorized_keys", fileNonEmpty("~/.ssh/authorized_keys"),
          hint="authorized_keys is missing or empty, so passwordless login cannot work. "
               "Re-run Part 5 on your own computer: "
               "ssh-copy-id ${DEVICE_USER}@${DEVICE_ADDRESS} - or append the key by "
               "hand as described in the Part 5 fallback note.",
          link="https://www.ssh.com/academy/ssh/authorized-keys-file",
          linkText="authorized_keys explained"),
], successNote="Your laptop's public key is installed on the Thor - passwordless "
               "login works. Parts 1-5 complete.",
   docLink="https://www.ssh.com/academy/ssh/public-key-authentication",
   docLinkText="How public-key authentication works")

                        checkpoint - Parts 1-5 - SSH key reaches the Thor                        
                                                                                                 
  check                                   result     detail                                      
 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 
  ~/.ssh exists on the Thor               ✓ ok       /Users/papka/.ssh                           
  your public key is in authorized_keys   ✗ failed   missing: /Users/papka/.ssh/authorized_keys

False

---
## Part 6 · Make an SSH Shortcut with a Config File

Typing the full SSH command gets repetitive. An SSH config file on your computer creates a short alias.

**[Your computer · Terminal]** Create or edit your SSH config file (`nano ~/.ssh/config`) and add an entry for the Thor. Replace `<thor-address>` and `<your-account>` with the same values you used for `DEVICE_ADDRESS` and `DEVICE_USER` (a config file takes literal values, not shell variables):

```text
Host thor
    HostName <thor-address>
    User <your-account>
    IdentityFile ~/.ssh/id_ed25519
```

Set safe permissions, then connect with the short name:

```bash
chmod 600 ~/.ssh/config
ssh thor
exit
```

You can also copy files using the short name. Create a local test file and copy it to the Thor:

```bash
echo "copied with scp" > ~/edgeToolingLab/localCopyTest.txt
scp ~/edgeToolingLab/localCopyTest.txt thor:~/
```

SSH back in and check the copied file (or just run the checkpoint below — the notebook is already on the Thor):

```bash
ssh thor
cat ~/localCopyTest.txt
```

This shows how SSH configuration makes repeated remote work easier.

### Checkpoint · Part 6

The `scp` copy lands in your Thor home folder, where this notebook can verify it directly.

In [4]:
checkpoint("Part 6 - ssh config shortcut and scp", [
    check("scp'd test file arrived on the Thor", fileExists("~/localCopyTest.txt"),
          hint="Run the Part 6 scp command on your own computer: "
               "scp ~/edgeToolingLab/localCopyTest.txt thor:~/ - if 'thor' is not "
               "recognized, re-check your ~/.ssh/config entry on your laptop.",
          link="https://man.openbsd.org/scp",
          linkText="scp man page"),
    check("test file has the expected content",
          fileContains("~/localCopyTest.txt", "copied with scp"),
          hint="The file exists but its content differs - recreate it on your laptop with "
               "echo \"copied with scp\" > ~/edgeToolingLab/localCopyTest.txt and scp it again.",
          link="https://man.openbsd.org/ssh_config",
          linkText="ssh_config man page"),
], successNote="The 'thor' shortcut and scp both work - laptop-side setup is done.",
   docLink="https://man.openbsd.org/ssh_config",
   docLinkText="ssh_config man page")

                     checkpoint - Part 6 - ssh config shortcut and scp                      
                                                                                            
  check                                 result     detail                                   
 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 
  scp'd test file arrived on the Thor   ✗ failed   missing: /Users/papka/localCopyTest.txt  
  test file has the expected content    ✗ failed   missing: /Users/papka/localCopyTest.txt

False

---
# You Are Now Working On the Thor

Everything from here to the end of the lab runs **on the Jetson Thor**, not on your own computer. In the terminal version of this lab you would now `ssh thor` and stay connected — this notebook already runs on the Thor through JupyterLab, so from here on every step is an ordinary **[Notebook cell]**.

> Environment variables you set on your computer (like `DEVICE_ADDRESS`) do **not** carry over an SSH connection — and they do not exist in this notebook's kernel either. Any variable you need on the Thor must be set on the Thor.

**[Notebook cell]** Confirm where you are — `hostname` should print the Thor's name:

In [5]:
!whoami
!hostname

papka
MBPro2024-3.local


---
## Part 7 · Practice Linux Commands on the Thor

Create your lab folder on the Thor, then practice the core command families. One notebook quirk: every `!` line is its own tiny shell, so a bare `cd` would not persist — the notebook uses `%cd` for real directory changes and `cd ~ && pwd`-style one-liners to demonstrate navigation.

In [6]:
!mkdir -p ~/edgeToolingLab
%cd ~/edgeToolingLab

/Users/papka/edgeToolingLab


**[Notebook cell]** Practice navigation commands:

In [7]:
!pwd
!ls
!ls -la
!cd ~ && pwd     # this cd lasts only for its own line...
!pwd             # ...so the notebook is still in the lab folder

/Users/papka/edgeToolingLab
total 0
drwxr-xr-x@   2 papka  staff    64 Jul 10 15:00 .
drwx------@ 147 papka  staff  4704 Jul 10 15:00 ..
/Users/papka
/Users/papka/edgeToolingLab


**[Notebook cell]** Practice file commands:

In [8]:
!touch notes.txt
!echo "Edge Tooling Lab" > notes.txt
!cat notes.txt
!cp notes.txt notesCopy.txt
!mv notesCopy.txt backupNotes.txt
!ls -la

Edge Tooling Lab
total 16
drwxr-xr-x@   4 papka  staff   128 Jul 10 15:01 .
drwx------@ 147 papka  staff  4704 Jul 10 15:00 ..
-rw-r--r--@   1 papka  staff    17 Jul 10 15:00 backupNotes.txt
-rw-r--r--@   1 papka  staff    17 Jul 10 15:00 notes.txt


**[Notebook cell]** Practice directory commands:

In [9]:
!mkdir -p logs
!mkdir -p scripts
!ls -la

total 16
drwxr-xr-x@   6 papka  staff   192 Jul 10 15:01 .
drwx------@ 147 papka  staff  4704 Jul 10 15:00 ..
-rw-r--r--@   1 papka  staff    17 Jul 10 15:00 backupNotes.txt
drwxr-xr-x@   2 papka  staff    64 Jul 10 15:01 logs
-rw-r--r--@   1 papka  staff    17 Jul 10 15:00 notes.txt
drwxr-xr-x@   2 papka  staff    64 Jul 10 15:01 scripts


**[Notebook cell]** Practice viewing system information:

In [10]:
!whoami
!hostname
!date
!uname -a
!df -h
!free -h

papka
MBPro2024-3.local
Fri Jul 10 15:01:02 CDT 2026
Darwin MBPro2024-3.local 25.5.0 Darwin Kernel Version 25.5.0: Tue Jun  9 22:28:34 PDT 2026; root:xnu-12377.121.10~1/RELEASE_ARM64_T6041 arm64
Filesystem        Size    Used   Avail Capacity iused ifree %iused  Mounted on
/dev/disk3s1s1   3.6Ti    12Gi   404Gi     3%    459k  4.2G    0%   /
devfs            417Ki   417Ki     0Bi   100%    1.4k     0  100%   /dev
/dev/disk3s6     3.6Ti    20Ki   404Gi     1%       0  4.2G    0%   /System/Volumes/VM
/dev/disk3s2     3.6Ti   8.4Gi   404Gi     3%    1.5k  4.2G    0%   /System/Volumes/Preboot
/dev/disk3s4     3.6Ti   6.5Mi   404Gi     1%     108  4.2G    0%   /System/Volumes/Update
/dev/disk1s2     500Mi   6.0Mi   481Mi     2%       1  4.9M    0%   /System/Volumes/xarts
/dev/disk1s1     500Mi   5.6Mi   481Mi     2%      32  4.9M    0%   /System/Volumes/iSCPreboot
/dev/disk1s3     500Mi   2.3Mi   481Mi     1%      87  4.9M    0%   /System/Volumes/Hardware
/dev/disk3s5     3.6Ti   3.2Ti   40

**[Notebook cell]** Practice searching and reading:

In [11]:
!echo "remote login successful" >> notes.txt
!echo "git tracks changes" >> notes.txt
!cat notes.txt
!grep "git" notes.txt
!tail notes.txt

Edge Tooling Lab
remote login successful
git tracks changes
git tracks changes
Edge Tooling Lab
remote login successful
git tracks changes


This shows how Linux commands inspect and control a remote device through SSH.

### Checkpoint · Part 7

In [12]:
checkpoint("Part 7 - Linux command practice", [
    check("notes.txt written and appended",
          fileContains("~/edgeToolingLab/notes.txt", "git tracks changes"),
          hint="Re-run the 'file commands' cell and then the 'searching and reading' "
               "cell in Part 7 - the appends (>>) add the missing lines.",
          link="https://www.gnu.org/software/bash/manual/html_node/Redirections.html",
          linkText="Shell redirection manual"),
    check("copy/rename produced backupNotes.txt",
          fileExists("~/edgeToolingLab/backupNotes.txt"),
          hint="Re-run the 'file commands' cell - cp makes notesCopy.txt and mv "
               "renames it to backupNotes.txt.",
          link="https://man7.org/linux/man-pages/man1/mv.1.html",
          linkText="mv man page"),
    check("logs/ folder created", dirExists("~/edgeToolingLab/logs"),
          hint="Re-run the 'directory commands' cell (mkdir -p logs).",
          link="https://man7.org/linux/man-pages/man1/mkdir.1.html",
          linkText="mkdir man page"),
    check("scripts/ folder created", dirExists("~/edgeToolingLab/scripts"),
          hint="Re-run the 'directory commands' cell (mkdir -p scripts).",
          link="https://man7.org/linux/man-pages/man1/mkdir.1.html",
          linkText="mkdir man page"),
], successNote="Core Linux navigation, file, and directory commands all practiced.",
   docLink="https://www.gnu.org/software/coreutils/manual/coreutils.html",
   docLinkText="GNU coreutils manual")

                         checkpoint - Part 7 - Linux command practice                          
                                                                                               
  check                                  result   detail                                       
 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 
  notes.txt written and appended         ✓ ok     'git tracks changes' found in notes.txt      
  copy/rename produced backupNotes.txt   ✓ ok     /Users/papka/edgeToolingLab/backupNotes.txt  
  logs/ folder created                   ✓ ok     /Users/papka/edgeToolingLab/logs             
  scripts/ folder created                ✓ ok     /Users/papka/edgeToolingLab/scripts

True

---
## Part 8 · Create a Simple Project on the Thor

**[Notebook cell]** Create a simple project folder inside your lab folder and move into it:

In [13]:
!mkdir -p ~/edgeToolingLab/deviceStatus/scripts
%cd ~/edgeToolingLab/deviceStatus

/Users/papka/edgeToolingLab/deviceStatus


**[Notebook cell]** Create `README.md` (the original lab uses `nano`; here the notebook writes the file for you):

In [14]:
%%writefile README.md
# Device Status Project

This project was created on a Jetson Thor edge device over SSH.

It records basic device information and demonstrates a simple Git workflow.

Writing README.md


In [15]:
# Preview the README we just wrote.
showFile("~/edgeToolingLab/deviceStatus/README.md")

**[Notebook cell]** Create `scripts/deviceStatus.sh`:

In [16]:
%%writefile scripts/deviceStatus.sh
#!/bin/bash

echo "Device status report"
echo "User: $(whoami)"
echo "Host: $(hostname)"
echo "Date: $(date)"
echo ""
echo "Disk usage:"
df -h /
echo ""
echo "Memory usage:"
free -h

Writing scripts/deviceStatus.sh


In [17]:
# Preview the status script we just wrote.
showFile("~/edgeToolingLab/deviceStatus/scripts/deviceStatus.sh")

**[Notebook cell]** Make the script executable and run it:

In [18]:
!chmod +x scripts/deviceStatus.sh
!./scripts/deviceStatus.sh

Device status report
User: papka
Host: MBPro2024-3.local
Date: Fri Jul 10 15:01:05 CDT 2026

Disk usage:
Filesystem        Size    Used   Avail Capacity iused ifree %iused  Mounted on
/dev/disk3s1s1   3.6Ti    12Gi   404Gi     3%    459k  4.2G    0%   /

Memory usage:
./scripts/deviceStatus.sh: line 12: free: command not found


**[Notebook cell]** Save the output to a log file:

In [19]:
!mkdir -p logs
!./scripts/deviceStatus.sh > logs/status.txt
!cat logs/status.txt

./scripts/deviceStatus.sh: line 12: free: command not found
Device status report
User: papka
Host: MBPro2024-3.local
Date: Fri Jul 10 15:01:06 CDT 2026

Disk usage:
Filesystem        Size    Used   Avail Capacity iused ifree %iused  Mounted on
/dev/disk3s1s1   3.6Ti    12Gi   404Gi     3%    459k  4.2G    0%   /

Memory usage:


The project now has source files and generated output.

### Checkpoint · Part 8

In [20]:
checkpoint("Part 8 - device status project created", [
    check("README.md written",
          fileContains("~/edgeToolingLab/deviceStatus/README.md", "Device Status Project"),
          hint="Re-run the %%writefile README.md cell in Part 8.",
          link="https://www.markdownguide.org/basic-syntax/",
          linkText="Markdown basics"),
    check("deviceStatus.sh exists",
          fileExists("~/edgeToolingLab/deviceStatus/scripts/deviceStatus.sh"),
          hint="Re-run the %%writefile scripts/deviceStatus.sh cell in Part 8.",
          link="https://www.gnu.org/software/bash/manual/bash.html",
          linkText="Bash manual"),
    check("deviceStatus.sh runs cleanly",
          commandSucceeds("bash ~/edgeToolingLab/deviceStatus/scripts/deviceStatus.sh"),
          hint="The script exists but errors when run - re-run the %%writefile cell to "
               "restore its content, then chmod +x it again.",
          link="https://www.gnu.org/software/bash/manual/bash.html",
          linkText="Bash manual"),
    check("status log captured",
          fileNonEmpty("~/edgeToolingLab/deviceStatus/logs/status.txt", minLines=5),
          hint="Re-run the 'save the output to a log file' cell - it redirects the "
               "script output into logs/status.txt.",
          link="https://www.gnu.org/software/bash/manual/html_node/Redirections.html",
          linkText="Shell redirection manual"),
], successNote="Project skeleton, runnable script, and captured log all in place.",
   docLink="https://man7.org/linux/man-pages/man1/chmod.1.html",
   docLinkText="chmod man page")

                             checkpoint - Part 8 - device status project created                              
                                                                                                              
  check                          result     detail                                                            
 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 
  README.md written              ✓ ok       'Device Status Project' found in README.md                        
  deviceStatus.sh exists         ✓ ok       /Users/papka/edgeToolingLab/deviceStatus/scripts/deviceStatus.sh  
  deviceStatus.sh runs cleanly   ✗ failed   Device status report                                              
  status log captured            ✓ ok       status.txt: 10 line(s)

False

---
## Part 9 · Learn How Git Tracks Changes

**[Notebook cell]** Check that Git is installed:

In [21]:
!git --version

git version 2.54.0


**[Notebook cell]** Set your Git identity on the Thor. **Edit the two values first**, then run the cell — Git records them in every commit:

In [22]:
# EDIT these two values with your information, then run the cell.
studentName = "Student Name"
studentEmail = "student@example.com"

!git config --global user.name "{studentName}"
!git config --global user.email "{studentEmail}"
!git config --global --list

user.name=Student Name
user.email=student@example.com
core.editor=code --wait
core.autocrlf=input
init.defaultbranch=main
pull.rebase=false
push.default=simple
push.followtags=true
merge.conflictstyle=diff3
diff.colormoved=default
filter.lfs.clean=git-lfs clean -- %f
filter.lfs.smudge=git-lfs smudge -- %f
filter.lfs.process=git-lfs filter-process
filter.lfs.required=true
alias.st=status
alias.co=checkout
alias.cm=commit -m
alias.cam=commit -am
alias.pl=pull
alias.ps=push
alias.log1=log --oneline --graph --decorate
alias.undo=reset --soft HEAD~1


**[Notebook cell]** Initialize a Git repository in the project folder and check its status — Git should show untracked files:

In [23]:
%cd ~/edgeToolingLab/deviceStatus
!git init
!git status

/Users/papka/edgeToolingLab/deviceStatus
Initialized empty Git repository in /Users/papka/edgeToolingLab/deviceStatus/.git/
On branch main

No commits yet

Untracked files:
  (use "git add <file>..." to include in what will be committed)
	README.md
	logs/
	scripts/

nothing added to commit but untracked files present (use "git add" to track)


**[Notebook cell]** Add files to the staging area and check the status again:

In [24]:
!git add README.md scripts/deviceStatus.sh
!git status

On branch main

No commits yet

Changes to be committed:
  (use "git rm --cached <file>..." to unstage)
	new file:   README.md
	new file:   scripts/deviceStatus.sh

Untracked files:
  (use "git add <file>..." to include in what will be committed)
	logs/



**[Notebook cell]** Commit the files and view the history:

In [25]:
!git commit -m "Create device status project"
!git log --oneline

[main (root-commit) 5d0be91] Create device status project
 2 files changed, 17 insertions(+)
 create mode 100644 README.md
 create mode 100755 scripts/deviceStatus.sh
5d0be91 (HEAD -> main) Create device status project


This shows the basic Git cycle:

```text
edit files -> git add -> git commit -> project history
```

### Checkpoint · Part 9

In [26]:
import os
projectDir = os.path.expanduser("~/edgeToolingLab/deviceStatus")
checkpoint("Part 9 - first Git commit", [
    check("git identity configured",
          outputContains(["git", "config", "--global", "--list"], "user.email="),
          hint="Edit studentName / studentEmail in the identity cell of Part 9 and "
               "re-run it - commits need an author.",
          link="https://git-scm.com/docs/git-config",
          linkText="git config docs"),
    check("repository initialized with a commit",
          gitRepoAt("~/edgeToolingLab/deviceStatus", minCommits=1),
          hint="Re-run the git init, git add, and git commit cells in Part 9, in order.",
          link="https://git-scm.com/docs/git-init",
          linkText="git init docs"),
    check("commit message recorded",
          outputContains(["git", "-C", projectDir, "log", "--oneline"],
                         "Create device status project"),
          hint="The commit is missing - re-run the git add and git commit cells in Part 9.",
          link="https://git-scm.com/docs/git-commit",
          linkText="git commit docs"),
], successNote="Your project history has begun - one commit recorded on the Thor.",
   docLink="https://git-scm.com/book/en/v2/Git-Basics-Recording-Changes-to-the-Repository",
   docLinkText="Git basics: recording changes")

                              checkpoint - Part 9 - first Git commit                              
                                                                                                  
  check                                  result   detail                                          
 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 
  git identity configured                ✓ ok     'user.email=' found in output                   
  repository initialized with a commit   ✓ ok     1 commit(s)                                     
  commit message recorded                ✓ ok     'Create device status project' found in output

True

---
## Part 10 · Track a Change with Git

You already know the add → commit cycle from Part 9. Here you practice it once more and meet two new ideas: seeing a change with `git diff`, and telling Git to *ignore* files.

**[Notebook cell]** Edit `README.md` (the original lab uses `nano`; here the cell appends the new line), then review the change with `git diff`:

In [27]:
%cd ~/edgeToolingLab/deviceStatus
!echo "The script can be run after logging in with SSH." >> README.md
!git diff

/Users/papka/edgeToolingLab/deviceStatus
diff --git a/README.md b/README.md
index 7de5839..bc5b30c 100644
--- a/README.md
+++ b/README.md
@@ -3,3 +3,4 @@
 This project was created on a Jetson Thor edge device over SSH.
 
 It records basic device information and demonstrates a simple Git workflow.
+The script can be run after logging in with SSH.


**[Notebook cell]** Stage and commit the change:

In [28]:
!git add README.md
!git commit -m "Document SSH usage" 

[main b78cce6] Document SSH usage
 1 file changed, 1 insertion(+)


**[Notebook cell]** Now create a `.gitignore` file:

In [29]:
%%writefile .gitignore
logs/
*.log

Writing .gitignore


**[Notebook cell]** Check the status — `logs/` no longer shows as untracked — then add and commit `.gitignore`:

In [30]:
!git status
!git add .gitignore
!git commit -m "Ignore generated logs" 

On branch main
Untracked files:
  (use "git add <file>..." to include in what will be committed)
	.gitignore

nothing added to commit but untracked files present (use "git add" to track)
[main 7169344] Ignore generated logs
 1 file changed, 2 insertions(+)
 create mode 100644 .gitignore


This shows that Git should track source files and documentation, but not every generated file.

### Checkpoint · Part 10

In [31]:
import os
projectDir = os.path.expanduser("~/edgeToolingLab/deviceStatus")
checkpoint("Part 10 - diff and .gitignore", [
    check(".gitignore ignores the logs folder",
          fileContains("~/edgeToolingLab/deviceStatus/.gitignore", "logs/"),
          hint="Re-run the %%writefile .gitignore cell in Part 10.",
          link="https://git-scm.com/docs/gitignore",
          linkText="gitignore docs"),
    check("git actually ignores logs/status.txt",
          outputContains(["git", "-C", projectDir, "check-ignore", "logs/status.txt"],
                         "logs/status.txt"),
          hint="git check-ignore did not match - make sure .gitignore contains the "
               "line 'logs/' and was saved in the project folder.",
          link="https://git-scm.com/docs/git-check-ignore",
          linkText="git check-ignore docs"),
    check("three commits recorded",
          gitRepoAt("~/edgeToolingLab/deviceStatus", minCommits=3),
          hint="Re-run the commit cells in Parts 9 and 10 - you should have "
               "'Create device status project', 'Document SSH usage', and "
               "'Ignore generated logs'.",
          link="https://git-scm.com/docs/git-log",
          linkText="git log docs"),
    check("SSH-usage commit present",
          outputContains(["git", "-C", projectDir, "log", "--oneline"],
                         "Ignore generated logs"),
          hint="Re-run the 'add and commit .gitignore' cell in Part 10.",
          link="https://git-scm.com/docs/git-commit",
          linkText="git commit docs"),
], successNote="You can now see changes before committing and keep generated files out "
               "of history.",
   docLink="https://git-scm.com/book/en/v2/Git-Basics-Recording-Changes-to-the-Repository",
   docLinkText="Git basics: recording changes")

                        checkpoint - Part 10 - diff and .gitignore                         
                                                                                           
  check                                  result   detail                                   
 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 
  .gitignore ignores the logs folder     ✓ ok     'logs/' found in .gitignore              
  git actually ignores logs/status.txt   ✓ ok     'logs/status.txt' found in output        
  three commits recorded                 ✓ ok     3 commit(s)                              
  SSH-usage commit present               ✓ ok     'Ignore generated logs' found in output

True

---
## Part 11 · Connect the Thor to GitHub over SSH

Git can use SSH to connect to GitHub. A GitHub SSH address looks like this:

```text
git@github.com:username/repository.git
```

The Thor needs its **own** SSH key to talk to GitHub (separate from the laptop key in Part 4).

**[Notebook cell]** Check whether one already exists:

In [32]:
!ls -la ~/.ssh

total 224
drwx------@  24 papka  staff    768 Jul 10 06:31 .
drwx------@ 147 papka  staff   4704 Jul 10 15:01 ..
drwx------@  11 papka  staff    352 Jul 10 06:31 agent
-rw-------@   1 papka  staff    464 Apr 22 12:52 aspark00
-rw-------@   1 papka  staff     96 Apr 22 12:50 aspark00.pub
-rw-------@   1 papka  staff    419 Apr 22 14:08 asparkNvsync
-rw-r--r--@   1 papka  staff    109 Apr 22 14:08 asparkNvsync.pub
-rw-------@   1 papka  staff   3586 Mar 29 07:25 config
-rw-------@   1 papka  staff   3743 Apr 22 13:16 config~
-rwxr-xr-x@   1 papka  staff     89 Feb 15 13:33 cpssh.sh
-rwxr-xr-x@   1 papka  staff    158 Feb 15 13:29 cpssh.sh~
srw-------@   1 papka  staff      0 Jul 10 06:31 ctrl-core@localhost:58933
-rw-------@   1 papka  staff    464 Apr 22 14:09 id_ed25519
-rw-------@   1 papka  staff    464 Apr 22 14:07 id_ed25519.preNvsync
-rw-r--r--@   1 papka  staff    109 Apr 22 14:09 id_ed25519.pub
-rw-------@   1 papka  staff     96 Apr 22 14:07 id_ed25519.pub.preNvsync
-rw-------@

**[Notebook cell]** Create a key on the Thor only if it does not exist yet. The original lab runs `ssh-keygen` interactively; this cell passes `-N ""` (no passphrase) so it can run non-interactively — if your instructor wants a passphrase, run `ssh-keygen -t ed25519 -C "${USER}-thor-github-key"` in a **[Terminal]** instead. Then view the public key:

In [33]:
![ -f ~/.ssh/id_ed25519 ] || ssh-keygen -t ed25519 -N "" -f ~/.ssh/id_ed25519 -C "${USER}-thor-github-key"
!cat ~/.ssh/id_ed25519.pub

ssh-ed25519 AAAAC3NzaC1lZDI1NTE5AAAAII5wwcP5oFD5YnqgWpbHVGhlPgVpZ9IHIhMia+icSStC papka@MBPro2024-unified-key


Copy the public key printed above (the whole `ssh-ed25519 ...` line) and add it to your GitHub account. In GitHub, the path is:

```text
Settings -> SSH and GPG keys -> New SSH key
```

Paste the public key and save it.

**[Notebook cell]** Test the SSH connection to GitHub. The original lab asks you to type `yes` at the first-connection host prompt; here `-o StrictHostKeyChecking=accept-new` answers it for you. GitHub greets you and then closes the connection, so `ssh` exits non-zero — the `|| true` keeps that expected exit from looking like a failure:

In [34]:
!ssh -o StrictHostKeyChecking=accept-new -T git@github.com || true

Hi mpapka! You've successfully authenticated, but GitHub does not provide shell access.


**You should see:** a greeting like `Hi <your-username>! You've successfully authenticated, but GitHub does not provide shell access.` That sentence *is* the success message — GitHub closing the connection right after is expected and not an error.

Important:

```text
SSH authentication proves who you are.
Git authorization decides which repositories you can access.
```

Your SSH key can work, but you still need permission to push to a specific repository.

### Checkpoint · Part 11

In [35]:
checkpoint("Part 11 - Thor authenticated to GitHub", [
    check("Thor has an SSH key pair", fileExists("~/.ssh/id_ed25519.pub"),
          hint="Re-run the key-creation cell in Part 11 (or run ssh-keygen in a Terminal).",
          link="https://docs.github.com/en/authentication/connecting-to-github-with-ssh/generating-a-new-ssh-key-and-adding-it-to-the-ssh-agent",
          linkText="GitHub: generating an SSH key"),
    check("GitHub accepts the Thor's key",
          outputContains(["ssh", "-o", "StrictHostKeyChecking=accept-new",
                          "-T", "git@github.com"], "successfully authenticated"),
          hint="GitHub did not recognize the key - add ~/.ssh/id_ed25519.pub to your "
               "GitHub account (Settings -> SSH and GPG keys -> New SSH key), then "
               "re-run this checkpoint.",
          link="https://docs.github.com/en/authentication/connecting-to-github-with-ssh/adding-a-new-ssh-key-to-your-github-account",
          linkText="GitHub: adding an SSH key"),
], successNote="The Thor can authenticate to GitHub over SSH - pushing is now possible.",
   docLink="https://docs.github.com/en/authentication/connecting-to-github-with-ssh/testing-your-ssh-connection",
   docLinkText="GitHub: testing your SSH connection")

                   checkpoint - Part 11 - Thor authenticated to GitHub                   
                                                                                         
  check                           result   detail                                        
 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 
  Thor has an SSH key pair        ✓ ok     /Users/papka/.ssh/id_ed25519.pub              
  GitHub accepts the Thor's key   ✓ ok     'successfully authenticated' found in output

True

---
## Part 12 · Push the Project to GitHub

In a browser, create an **empty** repository on GitHub. Example repository name:

```text
deviceStatusLab
```

Do **not** initialize it with a README, because the project already has one.

**[Notebook cell]** Set the remote repository URL. **Edit `githubUser` first** — the cell removes any earlier `origin` so it is safe to re-run:

In [36]:
# EDIT this to your GitHub username, then run the cell.
githubUser = "YOUR-GITHUB-USERNAME"

%cd ~/edgeToolingLab/deviceStatus
!git remote remove origin 2>/dev/null || true
!git remote add origin git@github.com:{githubUser}/deviceStatusLab.git
!git remote -v

/Users/papka/edgeToolingLab/deviceStatus
origin	git@github.com:YOUR-GITHUB-USERNAME/deviceStatusLab.git (fetch)
origin	git@github.com:YOUR-GITHUB-USERNAME/deviceStatusLab.git (push)


**[Notebook cell]** Rename the branch to `main` and push the project:

In [37]:
!git branch -M main
!git push -u origin main

ERROR: Repository not found.
fatal: Could not read from remote repository.

Please make sure you have the correct access rights
and the repository exists.


Refresh the repository page in the browser. You should see:

- `README.md`
- `scripts/deviceStatus.sh`
- `.gitignore`
- commit history

This shows Git using SSH as the secure transport to send commits to GitHub.

### Checkpoint · Part 12

In [38]:
import os
projectDir = os.path.expanduser("~/edgeToolingLab/deviceStatus")
checkpoint("Part 12 - project pushed to GitHub", [
    check("remote uses the SSH URL",
          outputContains(["git", "-C", projectDir, "remote", "-v"], "git@github.com"),
          hint="Edit githubUser in the remote cell of Part 12 and re-run it - the "
               "remote must be the SSH form git@github.com:USERNAME/deviceStatusLab.git.",
          link="https://git-scm.com/docs/git-remote",
          linkText="git remote docs"),
    check("branch renamed to main",
          outputContains(["git", "-C", projectDir, "rev-parse", "--abbrev-ref", "HEAD"],
                         "main"),
          hint="Re-run the 'git branch -M main' cell in Part 12.",
          link="https://git-scm.com/docs/git-branch",
          linkText="git branch docs"),
    check("push set the upstream (origin/main)",
          outputContains(["git", "-C", projectDir, "status", "-sb"], "origin/main"),
          hint="The push has not succeeded yet. Confirm the GitHub repository exists, "
               "your key passed the Part 11 checkpoint, and githubUser is spelled "
               "correctly - then re-run the push cell.",
          link="https://git-scm.com/docs/git-push",
          linkText="git push docs"),
], successNote="Your commits are on GitHub - refresh the repository page to see them.",
   docLink="https://docs.github.com/en/get-started/using-git/pushing-commits-to-a-remote-repository",
   docLinkText="GitHub: pushing commits")

                   checkpoint - Part 12 - project pushed to GitHub                    
                                                                                      
  check                                 result     detail                             
 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 
  remote uses the SSH URL               ✓ ok       'git@github.com' found in output   
  branch renamed to main                ✓ ok       'main' found in output             
  push set the upstream (origin/main)   ✗ failed   'origin/main' not found in output

False

---
## Part 13 · Clone the Repository Back to the Thor

Getting a project *from* GitHub onto an edge device is the other half of the workflow. Move out of the current copy, rename it, and clone a fresh copy over SSH.

**[Notebook cell]** The cell is guarded so re-running it is safe (`githubUser` comes from Part 12):

In [39]:
%cd ~/edgeToolingLab
![ -d deviceStatus ] && [ ! -d deviceStatus-original ] && mv deviceStatus deviceStatus-original || true
![ -d deviceStatusLab ] || git clone git@github.com:{githubUser}/deviceStatusLab.git
%cd ~/edgeToolingLab/deviceStatusLab

/Users/papka/edgeToolingLab
Cloning into 'deviceStatusLab'...
ERROR: Repository not found.
fatal: Could not read from remote repository.

Please make sure you have the correct access rights
and the repository exists.
[Errno 2] No such file or directory: '/Users/papka/edgeToolingLab/deviceStatusLab'
/Users/papka/edgeToolingLab


**[Notebook cell]** Confirm the files and history came with it:

In [40]:
!ls -la scripts
!git log --oneline

total 0
drwxr-xr-x@ 2 papka  staff   64 Jul 10 15:01 .
drwxr-xr-x@ 7 papka  staff  224 Jul 10 15:01 ..
fatal: not a git repository (or any of the parent directories): .git


This shows that Git repositories can move between devices while keeping their full history.

### Checkpoint · Part 13

In [41]:
checkpoint("Part 13 - fresh clone from GitHub", [
    check("original copy preserved", dirExists("~/edgeToolingLab/deviceStatus-original"),
          hint="Re-run the Part 13 clone cell - it renames deviceStatus to "
               "deviceStatus-original before cloning.",
          link="https://git-scm.com/docs/git-clone",
          linkText="git clone docs"),
    check("clone has the full history",
          gitRepoAt("~/edgeToolingLab/deviceStatusLab", minCommits=3),
          hint="The clone is missing or shallow - re-run the Part 13 clone cell "
               "(check githubUser and that Part 12's push succeeded).",
          link="https://git-scm.com/docs/git-clone",
          linkText="git clone docs"),
    check("script arrived with the clone",
          fileExists("~/edgeToolingLab/deviceStatusLab/scripts/deviceStatus.sh"),
          hint="scripts/deviceStatus.sh is not in the clone - confirm Part 12 pushed it "
               "(look at the repository page on GitHub), then delete "
               "~/edgeToolingLab/deviceStatusLab and re-run the clone cell.",
          link="https://git-scm.com/docs/git-clone",
          linkText="git clone docs"),
], successNote="A complete copy - files plus history - came back from GitHub.",
   docLink="https://git-scm.com/book/en/v2/Git-Basics-Getting-a-Git-Repository",
   docLinkText="Git basics: getting a repository")

                                  checkpoint - Part 13 - fresh clone from GitHub                                   
                                                                                                                   
  check                           result     detail                                                                
 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 
  original copy preserved         ✓ ok       /Users/papka/edgeToolingLab/deviceStatus-original                     
  clone has the full history      ✗ failed   no git repository at /Users/papka/edgeToolingLab/deviceStatusLab      
  script arrived with the clone   ✗ failed   missing:                                                              
                                             /Users/papka/edgeToolingLab/deviceStatusLab/scripts/deviceStatus.sh

False

---
## Part 14 · Practice Pulling and Pushing

**[Notebook cell]** Stay in the cloned repository and create a new file:

In [42]:
%cd ~/edgeToolingLab/deviceStatusLab

[Errno 2] No such file or directory: '/Users/papka/edgeToolingLab/deviceStatusLab'
/Users/papka/edgeToolingLab


In [43]:
%%writefile deviceNotes.md
# Device Notes

- SSH was used to connect to the Jetson Thor.
- Git was used to track project changes.
- An SSH key was used to connect the Thor to GitHub.

Writing deviceNotes.md


In [44]:
# Preview the notes file we just wrote.
showFile("~/edgeToolingLab/deviceStatusLab/deviceNotes.md")

**[Notebook cell]** Check status, then add, commit, push, and finally pull:

In [45]:
!git status
!git add deviceNotes.md
!git commit -m "Add device notes"
!git push
!git pull

fatal: not a git repository (or any of the parent directories): .git
fatal: not a git repository (or any of the parent directories): .git
fatal: not a git repository (or any of the parent directories): .git
fatal: not a git repository (or any of the parent directories): .git
fatal: not a git repository (or any of the parent directories): .git


This shows the normal Git collaboration cycle:

```text
git pull -> edit files -> git add -> git commit -> git push
```

### Checkpoint · Part 14

In [46]:
import os
cloneDir = os.path.expanduser("~/edgeToolingLab/deviceStatusLab")
checkpoint("Part 14 - pull/push cycle", [
    check("deviceNotes.md written",
          fileContains("~/edgeToolingLab/deviceStatusLab/deviceNotes.md",
                       "Device Notes"),
          hint="Re-run the %%writefile deviceNotes.md cell in Part 14.",
          link="https://www.markdownguide.org/basic-syntax/",
          linkText="Markdown basics"),
    check("notes committed",
          outputContains(["git", "-C", cloneDir, "log", "--oneline"], "Add device notes"),
          hint="Re-run the add/commit cell in Part 14.",
          link="https://git-scm.com/docs/git-commit",
          linkText="git commit docs"),
    check("clone in sync with origin/main",
          outputContains(["git", "-C", cloneDir, "status", "-sb"], "origin/main"),
          hint="The push did not complete - re-run the push cell and check the error "
               "message (permissions? network?).",
          link="https://git-scm.com/docs/git-push",
          linkText="git push docs"),
], successNote="Full collaboration cycle practiced: pull, edit, add, commit, push.",
   docLink="https://git-scm.com/book/en/v2/Git-Basics-Working-with-Remotes",
   docLinkText="Git basics: working with remotes")

                                      checkpoint - Part 14 - pull/push cycle                                       
                                                                                                                   
  check                            result     detail                                                               
 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 
  deviceNotes.md written           ✗ failed   missing: /Users/papka/edgeToolingLab/deviceStatusLab/deviceNotes.md  
  notes committed                  ✗ failed   'Add device notes' not found in output                               
  clone in sync with origin/main   ✗ failed   'origin/main' not found in output

False

---
## Part 15 · Troubleshoot Common SSH and Git Problems

**[Your computer · Terminal]** Back on your own computer, run SSH in verbose mode (use your literal account and the Thor's address — the `DEVICE_USER` / `DEVICE_ADDRESS` variables only exist in the shell where you exported them in Part 1):

```bash
ssh -v <your-account>@<thor-address>
```

Verbose mode shows more connection details.

Common SSH problems:

- wrong username
- wrong device address
- device is offline
- SSH service is not running
- private key file permissions are too open
- public key was not added to `authorized_keys`

Recommended SSH file permissions:

```text
~/.ssh                 -> 700
~/.ssh/id_ed25519      -> 600
~/.ssh/id_ed25519.pub  -> 644
~/.ssh/authorized_keys -> 600
~/.ssh/config          -> 600
```

**[Your computer · Terminal]** Fix common permissions:

```bash
chmod 700 ~/.ssh
chmod 600 ~/.ssh/id_ed25519
chmod 644 ~/.ssh/id_ed25519.pub
chmod 600 ~/.ssh/config 2>/dev/null || true
```

Common Git over SSH problems:

- the remote URL uses HTTPS instead of SSH
- the SSH key was not added to GitHub
- GitHub accepted the key, but the user lacks repository permission
- the repository name is misspelled
- the branch name is different from expected

**[Notebook cell]** Check (and if needed fix) the remote URL and re-test GitHub authentication:

In [47]:
!git -C ~/edgeToolingLab/deviceStatusLab remote -v
# If the remote is wrong, fix it (edit the username first):
# !git -C ~/edgeToolingLab/deviceStatusLab remote set-url origin git@github.com:{githubUser}/deviceStatusLab.git
!ssh -o StrictHostKeyChecking=accept-new -T git@github.com || true

fatal: cannot change to '/Users/papka/edgeToolingLab/deviceStatusLab': No such file or directory
Hi mpapka! You've successfully authenticated, but GitHub does not provide shell access.


This shows how SSH and Git troubleshooting often overlap — both depend on the same key and host identity checks. (Docker troubleshooting is covered in the next exercise, where you start using containers.)

---
## Key Terms

- **SSH (Secure Shell):** an encrypted connection that lets you log in to and run commands on a remote machine.
- **SSH key pair:** a private key (kept secret on your machine) and a public key (copied to servers/GitHub) that prove your identity without a password.
- **Jetson Thor:** the NVIDIA edge device this course runs on — a small computer with a built-in GPU for AI.
- **`${USER}`:** a shell variable holding your username, used to prefix names so your resources don't collide with other students' on the shared device.

---
## Cleanup

At the end of the hands-on exercise, remove only your own files and folders.

**[Notebook cell]** Optional — remove your lab folder on the Thor (uncomment to run). Your work is safe on GitHub after Part 12:

In [48]:
# !rm -rf ~/edgeToolingLab

**[Your computer · Terminal]** On your own computer, remove the local lab folder if you no longer need it:

```bash
rm -rf ~/edgeToolingLab
```

Do not remove SSH keys unless the instructor tells you to. Only remove your own files and folders.

### Lab scorecard

In [49]:
labSummary("Edge Development Tooling · SSH, Linux, Git, GitHub")

       Edge Development Tooling · SSH, Linux, Git, GitHub        
                                                                 
  checkpoint                               score   status        
 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 
  Parts 1-5 - SSH key reaches the Thor     1/2     ⚠ incomplete  
  Part 6 - ssh config shortcut and scp     0/2     ⚠ incomplete  
  Part 7 - Linux command practice          4/4     ✓ complete    
  Part 8 - device status project created   3/4     ⚠ incomplete  
  Part 9 - first Git commit                3/3     ✓ complete    
  Part 10 - diff and .gitignore            4/4     ✓ complete    
  Part 11 - Thor authenticated to GitHub   2/2     ✓ complete    
  Part 12 - project pushed to GitHub       2/3     ⚠ incomplete  
  Part 13 - fresh clone from GitHub        1/3     ⚠ incomplete  
  Part 14 - pull/push cycle                0/3     ⚠ incomplete

---
## Connect the Pieces

SSH, Linux, Git, and GitHub are used together in real edge projects:

```text
Your Computer
    | ssh
    v
Jetson Thor (edge device)
    | git pull
    v
Project Repository
```

A real project could use this setup to update code on an edge device, work on a shared class device, and push project history to GitHub. These are the foundation tools you will use in every lab that follows. In the **next exercise (System Architecture & Design)** you add the final piece — **Docker** — to package and run the edge services themselves on the Thor.